# LeetCode 848 — Shifting Letters bằng Python

Link đề: https://leetcode.com/problems/shifting-letters/ · Độ khó: **Medium** · Chủ đề: Array, String, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một chuỗi). Điểm cốt lõi: `shifts[i]` dịch **`i + 1` chữ cái đầu**, nên chữ cái ở vị trí `j` bị dịch bởi **tổng hậu tố** `shifts[j] + shifts[j+1] + … + shifts[n−1]` (mod 26). Notebook dùng oracle **mô phỏng đúng từng bước** (dịch từng lần một) để đối chiếu mọi cách.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi `i`, dịch `i + 1` chữ cái đầu của `s` thêm `shifts[i]` lần (mỗi lần dịch là chữ cái kế tiếp, `'z'` quay về `'a'`); trả về chuỗi cuối.
2. Nắm các công cụ Python: `ord`/`chr`, phép `%` trên **vòng tròn 26 chữ cái**, **tổng hậu tố** (suffix sum), **mảng hiệu (difference array)**, `itertools.accumulate`, `"".join`.
3. Đi từ **dịch từng tiền tố một (brute force, `O(n²)`)** → **tổng hậu tố duyệt ngược** → **mảng hiệu + tổng tiền tố** → **`accumulate` với "tổng trừ tiền tố"**.
4. Hiểu mấu chốt: ký tự thứ `j` chịu ảnh hưởng của mọi `shifts[i]` với `i ≥ j` (vì `i + 1 > j`); do phép dịch **cộng được** (dịch `a` rồi `b` = dịch `a + b`) và theo **modulo 26**, tổng độ dịch là `S_j = Σ_{i ≥ j} shifts[i]` rồi lấy `mod 26`.
5. Tránh các lỗi kinh điển: dịch ký tự `i` chỉ bằng `shifts[i]` (quên cộng dồn), cộng tiền tố thay vì **hậu tố**, quên `% 26` (ra ký tự ngoài `a..z`), nhầm hạn `i + 1` (lệch một), và tràn số nguyên ở ngôn ngữ có `int` 32-bit (tổng tới `10^14`).

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường và mảng số nguyên `shifts` cùng độ dài.

Gọi `shift()` của một chữ cái là chữ cái **kế tiếp** trong bảng chữ cái (vòng tròn: `'z'` thành `'a'`). Ví dụ `shift('a') = 'b'`, `shift('t') = 'u'`, `shift('z') = 'a'`.

Với mỗi `shifts[i] = x`, ta dịch **`i + 1` chữ cái đầu tiên** của `s` đi `x` lần.

Trả về chuỗi cuối cùng sau khi áp dụng mọi phép dịch.

**Ví dụ 1:** `s = "abc"`, `shifts = [3,5,9]` → `"rpl"`.

- Dịch 1 chữ cái đầu 3 lần: `"dbc"`.
- Dịch 2 chữ cái đầu 5 lần: `"igc"`.
- Dịch 3 chữ cái đầu 9 lần: `"rpl"`.

**Ví dụ 2:** `s = "aaa"`, `shifts = [1,2,3]` → `"gfd"`.

**Ràng buộc:** `1 <= s.length <= 10^5`; `s` gồm chữ cái thường; `shifts.length == s.length`; `0 <= shifts[i] <= 10^9`.

In [1]:
import random
import time
from itertools import accumulate, product
from typing import List

ex1 = ("abc", [3, 5, 9])
ex2 = ("aaa", [1, 2, 3])
for s, sh in (ex1, ex2):
    print(f"s = {s!r}, shifts = {sh}")

s = 'abc', shifts = [3, 5, 9]
s = 'aaa', shifts = [1, 2, 3]


## 3. Kiến thức nền

### 3.1 Chữ cái là số: `ord` và `chr`

`ord('a') = 97`, `ord('z') = 122`; `chr(97) = 'a'`. Chữ cái thứ `k` (đếm từ 0) trong bảng chữ cái có mã `97 + k`. Để làm toán, đổi chữ cái thành số `k = ord(c) − 97`.

In [2]:
print(ord("a"), ord("z"), chr(97), chr(122))
print([ord(c) - ord("a") for c in "abcxyz"])

97 122 a z
[0, 1, 2, 23, 24, 25]


### 3.2 Vòng tròn 26 chữ cái và phép `%`

Dịch chữ cái có chỉ số `k` đi `x` bước cho chữ cái chỉ số `(k + x) % 26`. Phép `%` giúp **quay vòng**: `(25 + 1) % 26 = 0` (`'z'` → `'a'`). Dịch `a` bước rồi `b` bước bằng dịch `(a + b)` bước (modulo 26): `(k + a + b) % 26 = ((k + a) % 26 + b) % 26`. Vì vậy với mỗi chữ cái ta chỉ cần biết **tổng số bước** (mod 26) — không cần dịch lần lượt.

In [3]:
def shift_char(c, x):
    return chr((ord(c) - 97 + x) % 26 + 97)


print(shift_char("a", 1), shift_char("z", 1), shift_char("t", 1), shift_char("c", 28), shift_char("c", 2))
assert shift_char(shift_char("x", 7), 9) == shift_char("x", 16)
# lớn: 10**9 bước = (10**9 % 26) bước
print("10^9 % 26 =", 10 ** 9 % 26, "| shift_char('a', 10**9) =", shift_char("a", 10 ** 9), "=", shift_char("a", 10 ** 9 % 26))

b a u e e
10^9 % 26 = 12 | shift_char('a', 10**9) = m = m


### 3.3 Tổng hậu tố (suffix sum) và tổng tiền tố (prefix sum)

- **Tổng tiền tố** `P[i] = shifts[0] + … + shifts[i]`.
- **Tổng hậu tố** `S[i] = shifts[i] + shifts[i+1] + … + shifts[n−1]`. Tính ngược: `S[n−1] = shifts[n−1]`, `S[i] = S[i+1] + shifts[i]`.

Liên hệ: `S[i] = P[n−1] − P[i−1]` (tổng toàn bộ trừ tiền tố trước `i`).

In [4]:
shifts = [3, 5, 9]
P = list(accumulate(shifts))
S = [0] * 3
run = 0
for i in range(2, -1, -1):
    run += shifts[i]
    S[i] = run
print("tiền tố:", P, "| hậu tố:", S, "| S[i] = tổng - P[i-1]:", [P[-1] - (P[i - 1] if i else 0) for i in range(3)])

tiền tố: [3, 8, 17] | hậu tố: [17, 14, 9] | S[i] = tổng - P[i-1]: [17, 14, 9]


### 3.4 Mảng hiệu (difference array)

Muốn "cộng `x` vào **mọi phần tử** trong đoạn `[l, r]`" nhiều lần, thay vì cộng từng phần tử ta ghi `diff[l] += x` và `diff[r + 1] -= x`; sau cùng lấy **tổng tiền tố** của `diff` để được giá trị thật. Bài này mỗi `shifts[i]` cộng vào đoạn `[0, i]`: `diff[0] += shifts[i]`, `diff[i + 1] -= shifts[i]`.

In [5]:
n = 5
diff = [0] * (n + 1)
for l, r, x in [(0, 2, 4), (1, 3, 3), (4, 4, 10)]:
    diff[l] += x
    diff[r + 1] -= x
print("diff:", diff)
print("tổng tiền tố của diff:", list(accumulate(diff[:n])))
brute = [0] * n
for l, r, x in [(0, 2, 4), (1, 3, 3), (4, 4, 10)]:
    for j in range(l, r + 1):
        brute[j] += x
print("cộng từng phần tử   :", brute)
assert list(accumulate(diff[:n])) == brute

diff: [4, 3, 0, -4, 7, -10]
tổng tiền tố của diff: [4, 7, 7, 3, 10]
cộng từng phần tử   : [4, 7, 7, 3, 10]


### 3.5 Kích thước tổng và tràn số nguyên

Mỗi `shifts[i]` tới `10^9` và có `10^5` phần tử: tổng hậu tố có thể tới `10^14`. Trong Python không tràn; ở C++/Java với `int` 32-bit **sẽ tràn** (cần `long`, hoặc lấy `% 26` **ngay trong khi cộng dồn**). Ta minh hoạ ở mục 13.

In [6]:
print("tổng lớn nhất:", 10 ** 5 * 10 ** 9, "| > 2^31 - 1 =", 2 ** 31 - 1, ":", 10 ** 5 * 10 ** 9 > 2 ** 31 - 1, "| < 2^63 - 1:", 10 ** 14 < 2 ** 63 - 1)

tổng lớn nhất: 100000000000000 | > 2^31 - 1 = 2147483647 : True | < 2^63 - 1: True


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 1: `s = "abc"`, `shifts = [3, 5, 9]`. Mỗi `shifts[i]` "phủ" các chữ cái đầu tới vị trí `i`:

```
shifts[0]=3 : dịch chữ 0
shifts[1]=5 : dịch chữ 0, 1
shifts[2]=9 : dịch chữ 0, 1, 2
```

Chữ cái `0` ('a') chịu cả ba phép: `3 + 5 + 9 = 17` bước → `'a' + 17 = 'r'`. Chữ cái `1` ('b') chịu hai: `5 + 9 = 14` → `'b' + 14 = 'p'`. Chữ cái `2` ('c') chỉ chịu `9` → `'c' + 9 = 'l'`. Kết quả `"rpl"` ✓.

Tổng quát: chữ thứ `j` chịu mọi `shifts[i]` với `i ≥ j` (vì phép thứ `i` phủ `i + 1` chữ đầu, tức vị trí `0..i`): tổng **hậu tố** `S[j]`.

In [7]:
s, shifts = ex1
S = [sum(shifts[j:]) for j in range(len(s))]
print("tổng hậu tố:", S)
print("kết quả:", "".join(shift_char(c, x) for c, x in zip(s, S)))
for j, c in enumerate(s):
    print(f"chữ {j} ({c!r}) chịu shifts[{j}:] = {shifts[j:]} = {S[j]} bước -> {shift_char(c, S[j])!r}")

tổng hậu tố: [17, 14, 9]
kết quả: rpl
chữ 0 ('a') chịu shifts[0:] = [3, 5, 9] = 17 bước -> 'r'
chữ 1 ('b') chịu shifts[1:] = [5, 9] = 14 bước -> 'p'
chữ 2 ('c') chịu shifts[2:] = [9] = 9 bước -> 'l'


## 5. Cách 1 — Brute Force: dịch từng tiền tố

**Ý tưởng:** làm đúng theo đề. Với mỗi `i`, dịch `i + 1` chữ cái đầu thêm `shifts[i]` bước — dùng `shifts[i] % 26` để tránh lặp `10^9` lần (dịch `x` bước tương đương dịch `x mod 26` bước).

In [8]:
def shifting_brute(s, shifts):
    a = list(s)
    for i, x in enumerate(shifts):
        for j in range(i + 1):
            a[j] = shift_char(a[j], x % 26)
    return "".join(a)


print(shifting_brute(*ex1), shifting_brute(*ex2))
assert shifting_brute(*ex1) == "rpl" and shifting_brute(*ex2) == "gfd"

rpl gfd


**Độ phức tạp:** hai vòng lồng nhau: `Σ (i + 1) = n(n+1)/2 ≈ 5·10^9` phép với `n = 10^5`: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** cùng một phép dịch được áp lại cho cả tiền tố dài. Mỗi chữ chỉ quan tâm **tổng** các phép dịch tác động lên nó.

## 6. Ý tưởng thuật toán / bước đột phá

Ký tự `j` chịu tác động của mọi `shifts[i]` với `i ≥ j`; vì dịch cộng được, độ dịch tổng của nó là `S[j] = Σ_{i ≥ j} shifts[i]` (mod 26). Tính mọi `S[j]` bằng **tổng hậu tố** trong `O(n)`: duyệt từ phải sang trái, giữ biến `total` cộng dồn.

Có ba cách viết tương đương: (a) **duyệt ngược** với `total`, (b) **mảng hiệu** (cộng `shifts[i]` vào đoạn `[0, i]`), (c) **`accumulate`**: `S[j] = tổng − tiền tố trước j`. Thêm `% 26` để số nhỏ (và tránh tràn ở ngôn ngữ khác).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** tổng hậu tố duyệt ngược, **Cách 3** mảng hiệu, **Cách 4** `accumulate` (tổng trừ tiền tố).

### Cách 2 — Tổng hậu tố duyệt ngược

**Giải thích từng đoạn:**

1. `total = 0`; duyệt `i` từ `n − 1` về `0`.
2. `total = (total + shifts[i]) % 26` (cộng `shifts[i]` vào mọi chữ có chỉ số `≤ i`).
3. Chữ thứ `i` nhận độ dịch `total`: `res[i] = shift_char(s[i], total)`.
4. Ghép kết quả.

In [9]:
def shifting_suffix(s, shifts):
    n = len(s)
    res = [""] * n
    total = 0
    for i in range(n - 1, -1, -1):
        total = (total + shifts[i]) % 26             # tổng hậu tố shifts[i:], lấy mod 26
        res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
    return "".join(res)


print(shifting_suffix(*ex1), shifting_suffix(*ex2))
assert shifting_suffix(*ex1) == "rpl" and shifting_suffix(*ex2) == "gfd"

rpl gfd


In [10]:
# Code nộp LeetCode cho Cách 2 (đáp án chính)
class SolutionSuffix:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        n = len(s)
        res = [""] * n
        total = 0
        for i in range(n - 1, -1, -1):
            total = (total + shifts[i]) % 26
            res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
        return "".join(res)

**Độ phức tạp:** một lượt `O(n)` thời gian; bộ nhớ `O(n)` cho kết quả. Lấy `% 26` mỗi bước giữ `total < 26 + 10^9` nên an toàn trong `int` 32-bit (`26 + 10^9 < 2^31`).

### Cách 3 — Mảng hiệu rồi tổng tiền tố

**Giải thích từng đoạn:**

1. `diff` độ dài `n + 1` toàn `0`. Với mỗi `i`: `diff[0] += shifts[i]` (đoạn `[0, i]` bắt đầu ở `0`) và `diff[i + 1] -= shifts[i]` (kết thúc sau `i`).
2. Tổng tiền tố của `diff` cho `shift_total[j]` = tổng các `shifts[i]` phủ `j` (tức `i ≥ j`).
3. Dịch từng chữ bằng `shift_total[j] % 26`.

(Với đoạn luôn bắt đầu từ `0`, `diff[0]` là tổng tất cả `shifts`; cách này tổng quát hoá cho **đoạn bất kỳ** như bài Shifting Letters II.)

In [11]:
def shifting_diff(s, shifts):
    n = len(s)
    diff = [0] * (n + 1)
    for i, x in enumerate(shifts):
        diff[0] += x                                  # đoạn [0, i]: bắt đầu tại 0
        diff[i + 1] -= x                              # và kết thúc sau i
    out = []
    cur = 0
    for j in range(n):
        cur += diff[j]                                # tổng tiền tố của diff = độ dịch của chữ j
        out.append(chr((ord(s[j]) - 97 + cur) % 26 + 97))
    return "".join(out)


print(shifting_diff(*ex1), shifting_diff(*ex2))
assert shifting_diff(*ex1) == "rpl" and shifting_diff(*ex2) == "gfd"

rpl gfd


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionDiff:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        n = len(s)
        diff = [0] * (n + 1)
        for i, x in enumerate(shifts):
            diff[0] += x
            diff[i + 1] -= x
        out, cur = [], 0
        for j in range(n):
            cur += diff[j]
            out.append(chr((ord(s[j]) - 97 + cur) % 26 + 97))
        return "".join(out)

**Độ phức tạp:** hai lượt `O(n)`; bộ nhớ `O(n)`. Các giá trị trung gian (`diff[0]` tới `10^14`) cần kiểu 64-bit ở C++/Java. Ưu điểm: dùng được cho bài **nhiều đoạn bất kỳ**.

### Cách 4 — `accumulate`: tổng trừ tiền tố

**Ý tưởng:** `S[j] = total − P[j − 1]` với `total = sum(shifts)` và `P` là tổng tiền tố. Dùng `itertools.accumulate` để lấy `P` gọn gàng: `prefix_before_j = [0] + P[:-1]`, rồi `S[j] = total − prefix_before_j[j]`.

In [13]:
def shifting_accumulate(s, shifts):
    total = sum(shifts)
    before = [0] + list(accumulate(shifts))[:-1]            # before[j] = shifts[0] + ... + shifts[j-1]
    return "".join(chr((ord(c) - 97 + total - b) % 26 + 97) for c, b in zip(s, before))


print(shifting_accumulate(*ex1), shifting_accumulate(*ex2))
assert shifting_accumulate(*ex1) == "rpl" and shifting_accumulate(*ex2) == "gfd"

rpl gfd


In [14]:
# Code nộp LeetCode cho Cách 4
class SolutionAccumulate:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        total = sum(shifts)
        before = [0] + list(accumulate(shifts))[:-1]
        return "".join(chr((ord(c) - 97 + total - b) % 26 + 97) for c, b in zip(s, before))

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Ngắn nhất, nhưng `total` có thể tới `10^14` (64-bit); `(… + total − b) % 26` dùng số dương nên Python `%` ra kết quả không âm; ở ngôn ngữ khác `total − b` luôn `≥ 0` nên cũng an toàn.

## 8. TRACE từng bước

**Trace Cách 2** với `s = "abc"`, `shifts = [3, 5, 9]` (duyệt ngược):

| `i` | `shifts[i]` | `total` trước | `total` sau `(total + shifts[i]) % 26` | `s[i]` | `s[i]` + `total` | `res[i]` |
|---|---|---|---|---|---|---|
| 2 | 9 | 0 | 9 | `c` (2) | `(2 + 9) % 26 = 11` | `l` |
| 1 | 5 | 9 | 14 | `b` (1) | `(1 + 14) % 26 = 15` | `p` |
| 0 | 3 | 14 | 17 | `a` (0) | `(0 + 17) % 26 = 17` | `r` |

Ghép `res = ['r', 'p', 'l']` → `"rpl"` ✓. Hàm trace in từng bước và minh hoạ đoạn có quay vòng `z → a`:

In [15]:
def trace_suffix(s, shifts):
    n = len(s)
    res = [""] * n
    total = 0
    print(f"s = {s!r}, shifts = {shifts}")
    print(f"{'i':>2} {'shifts[i]':>10} {'total':>8} {'chữ':>4} {'k':>3} {'(k+total)%26':>13} {'kết quả':>8}")
    for i in range(n - 1, -1, -1):
        total = (total + shifts[i]) % 26
        k = ord(s[i]) - 97
        new = (k + total) % 26
        res[i] = chr(new + 97)
        print(f"{i:>2} {shifts[i]:>10} {total:>8} {s[i]:>4} {k:>3} {new:>13} {res[i]:>8}")
    out = "".join(res)
    print("=>", out)
    return out


assert trace_suffix(*ex1) == "rpl"
print()
assert trace_suffix("zzz", [1, 26, 27]) == shifting_brute("zzz", [1, 26, 27])

s = 'abc', shifts = [3, 5, 9]
 i  shifts[i]    total  chữ   k  (k+total)%26  kết quả
 2          9        9    c   2            11        l
 1          5       14    b   1            15        p
 0          3       17    a   0            17        r
=> rpl

s = 'zzz', shifts = [1, 26, 27]
 i  shifts[i]    total  chữ   k  (k+total)%26  kết quả
 2         27        1    z  25             0        a
 1         26        1    z  25             0        a
 0          1        2    z  25             1        b
=> baa


**Trace Cách 3 (mảng hiệu)** với `shifts = [3, 5, 9]`, `n = 3`: `diff` bắt đầu `[0,0,0,0]`.

- `i = 0, x = 3`: `diff[0] += 3 → 3`; `diff[1] -= 3 → −3`.
- `i = 1, x = 5`: `diff[0] → 8`; `diff[2] → −5`.
- `i = 2, x = 9`: `diff[0] → 17`; `diff[3] → −9`.

`diff = [17, −3, −5, −9]`; tiền tố: `17, 14, 9` — đúng độ dịch của các chữ `a, b, c`. Hàm trace:

In [16]:
def trace_diff(s, shifts):
    n = len(s)
    diff = [0] * (n + 1)
    print("diff ban đầu:", diff)
    for i, x in enumerate(shifts):
        diff[0] += x
        diff[i + 1] -= x
        print(f"sau shifts[{i}]={x}: diff = {diff}")
    cur, shifted = 0, []
    for j in range(n):
        cur += diff[j]
        shifted.append(cur)
    print("tổng tiền tố của diff (độ dịch từng chữ):", shifted)
    out = "".join(chr((ord(s[j]) - 97 + shifted[j]) % 26 + 97) for j in range(n))
    print("=>", out)
    return out


assert trace_diff(*ex1) == "rpl"

diff ban đầu: [0, 0, 0, 0]
sau shifts[0]=3: diff = [3, -3, 0, 0]
sau shifts[1]=5: diff = [8, -3, -5, 0]
sau shifts[2]=9: diff = [17, -3, -5, -9]
tổng tiền tố của diff (độ dịch từng chữ): [17, 14, 9]
=> rpl


## 9. Code nộp LeetCode

Chọn **tổng hậu tố duyệt ngược** với `% 26` mỗi bước. Giữ nguyên `class Solution`, `shiftingLetters(self, s, shifts)`:

In [17]:
from typing import List


class Solution:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        n = len(s)
        res = [""] * n
        total = 0                                         # tổng hậu tố shifts[i:], giữ mod 26
        for i in range(n - 1, -1, -1):
            total = (total + shifts[i]) % 26              # chữ i chịu mọi shifts[j] với j >= i
            res[i] = chr((ord(s[i]) - ord("a") + total) % 26 + ord("a"))
        return "".join(res)


print(Solution().shiftingLetters("abc", [3, 5, 9]), Solution().shiftingLetters("aaa", [1, 2, 3]))   # rpl gfd
assert Solution().shiftingLetters(*ex1) == "rpl" and Solution().shiftingLetters(*ex2) == "gfd"

rpl gfd


## 10. Vì sao đúng

**(a) Chữ `j` chịu đúng các `shifts[i]` với `i ≥ j`.** Phép thứ `i` dịch `i + 1` chữ đầu, tức các vị trí `0, 1, …, i`. Chữ ở vị trí `j` thuộc đoạn đó khi và chỉ khi `j ≤ i`.

**(b) Dịch cộng được (modulo 26).** Đặt `k` là chỉ số chữ cái (`0..25`). Một lần dịch biến `k` thành `(k + 1) mod 26`. Dịch `x` lần cho `(k + x) mod 26` (quy nạp theo `x`). Áp dụng liên tiếp các phép dịch `x_1, x_2, …` cho `(k + x_1 + x_2 + …) mod 26` vì `((k + a) mod 26 + b) mod 26 = (k + a + b) mod 26`. Do đó thứ tự áp dụng không quan trọng và độ dịch tổng của chữ `j` là `S[j] = Σ_{i ≥ j} shifts[i]`, chữ cuối là `(k_j + S[j]) mod 26`.

**(c) Lấy `mod 26` giữa chừng đúng.** `(a + b) mod 26 = ((a mod 26) + (b mod 26)) mod 26`, nên `total = (total + shifts[i]) % 26` giữ nguyên giá trị `S[j] mod 26`.

**(d) Ba cách tương đương.** Cách 2 tính trực tiếp `S[j]` bằng truy hồi `S[j] = S[j+1] + shifts[j]`. Cách 3: tổng tiền tố của `diff` tại `j` bằng `Σ_i [j ∈ [0, i]]·shifts[i] = Σ_{i ≥ j} shifts[i]`. Cách 4: `S[j] = total − Σ_{i < j} shifts[i]`.

Ta kiểm chứng bằng máy: mọi cách khớp **mô phỏng từng bước** (dịch một chữ cái mỗi lần, không dùng modulo gộp) trên mọi đầu vào nhỏ (mục 12D) và khớp nhau trên đầu vào lớn (mục 12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dịch từng tiền tố | `O(n²)` | `O(n)` | oracle; `5·10^9` ở `n = 10^5` |
| 2. Tổng hậu tố duyệt ngược | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Mảng hiệu | `O(n)` | `O(n)` | dùng cho nhiều đoạn bất kỳ |
| 4. `accumulate` | `O(n)` | `O(n)` | ngắn nhất |

Đo thực tế (`n = 10^5`; brute chỉ `n = 3000`):

In [18]:
def random_case(rng, n, hi=10 ** 9):
    s = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    return s, [rng.randint(0, hi) for _ in range(n)]


def bench(fn, case, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*case)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(848)
big = random_case(rng, 10 ** 5)
ref = shifting_suffix(*big)
for name, fn in [("suffix", shifting_suffix), ("diff", shifting_diff), ("accumulate", shifting_accumulate)]:
    assert fn(*big) == ref
    print(f"n = 10^5 {name:11s} {bench(fn, big) * 1000:7.2f} ms")
mid = random_case(rng, 3000)
t = bench(shifting_brute, mid, 1)
print(f"brute n = 3000: {t * 1000:8.1f} ms -> n = 10^5 khoảng {t * (10 ** 5 / 3000) ** 2:.0f} s")

n = 10^5 suffix        10.60 ms


n = 10^5 diff          25.38 ms
n = 10^5 accumulate    18.06 ms


brute n = 3000:    436.4 ms -> n = 10^5 khoảng 485 s


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) vét cạn mọi đầu vào nhỏ với oracle **mô phỏng từng bước** (dịch một chữ cái mỗi lần, không dùng `%` gộp), (E) ngẫu nhiên có seed, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [19]:
def oracle(s, shifts):
    """Oracle mô phỏng ĐÚNG đề: với mỗi shifts[i] = x, dịch i+1 chữ đầu x lần, MỖI LẦN dịch một chữ cái kế tiếp
    (chỉ dùng cho x nhỏ). 'z' -> 'a' xử lý riêng, không dùng phép %."""
    a = list(s)
    nxt = {chr(c): chr(c + 1) for c in range(97, 122)}
    nxt["z"] = "a"
    for i, x in enumerate(shifts):
        for _ in range(x):
            for j in range(i + 1):
                a[j] = nxt[a[j]]
    return "".join(a)


IMPLS = {
    "brute": shifting_brute,
    "suffix": shifting_suffix,
    "diff": shifting_diff,
    "accumulate": shifting_accumulate,
    "Solution": lambda s, sh: Solution().shiftingLetters(s, list(sh)),
    "SolutionSuffix": lambda s, sh: SolutionSuffix().shiftingLetters(s, list(sh)),
    "SolutionDiff": lambda s, sh: SolutionDiff().shiftingLetters(s, list(sh)),
    "SolutionAccumulate": lambda s, sh: SolutionAccumulate().shiftingLetters(s, list(sh)),
}


def check(s, shifts, want=None, names=None):
    if want is None:
        want = oracle(s, shifts)
    for name in (names or IMPLS):
        got = IMPLS[name](s, list(shifts))
        assert got == want, (name, s, shifts, got, want)


check(*ex1, "rpl")
check(*ex2, "gfd")
assert oracle(*ex1) == "rpl" and oracle(*ex2) == "gfd"
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt và oracle mô phỏng từng bước đều đúng")

ví dụ 1, 2: cả 8 cài đặt và oracle mô phỏng từng bước đều đúng


### B. Edge case

- `n = 1`: chỉ một phép dịch.
- Mọi `shifts[i] = 0` → chuỗi giữ nguyên.
- Dịch đúng bội của `26` → chuỗi giữ nguyên (ví dụ `26`, `52`).
- Quay vòng `z → a`.
- Chuỗi toàn `z` và toàn `a`.
- `shifts[i] = 10^9` (tương đương `10^9 mod 26 = 12`).
- Chỉ `shifts` cuối khác `0`: tác động lên **mọi** chữ.
- Chỉ `shifts[0]` khác `0`: chỉ tác động lên chữ đầu.

In [20]:
cases = [
    ("a", [1], "b"), ("z", [1], "a"), ("a", [0], "a"), ("a", [26], "a"), ("a", [27], "b"),
    ("abc", [0, 0, 0], "abc"), ("abc", [26, 52, 78], "abc"),
    ("zzz", [1, 0, 0], "azz"), ("zzz", [0, 0, 1], "aaa"),
    ("abc", [1, 0, 0], "bbc"), ("abc", [0, 0, 5], "fgh"),
    ("xyz", [1, 1, 1], "zzz"[:0] + oracle("xyz", [1, 1, 1])),
]
for s, sh, want in cases:
    check(s, sh, want)
# giá trị lớn: 10^9 % 26 = 12 -> khớp "dịch 12 lần"
assert Solution().shiftingLetters("a", [10 ** 9]) == shift_char("a", 12) == "m"
assert Solution().shiftingLetters("abc", [10 ** 9] * 3) == "".join(shift_char(c, 12 * (3 - i)) for i, c in enumerate("abc"))
print(len(cases), "edge case +2 ca giá trị lớn: tất cả cài đặt đúng")

12 edge case +2 ca giá trị lớn: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_only_self(s, shifts):                          # chữ i chỉ dịch shifts[i] (quên cộng dồn)
    return "".join(shift_char(c, x % 26) for c, x in zip(s, shifts))


def bug_prefix(s, shifts):                             # cộng TIỀN TỐ thay vì HẬU TỐ
    out, total = [], 0
    for c, x in zip(s, shifts):
        total = (total + x) % 26
        out.append(shift_char(c, total))
    return "".join(out)


def bug_no_mod(s, shifts):                             # quên % 26: chr ra ký tự ngoài a..z
    res, total = [], 0
    for i in range(len(s) - 1, -1, -1):
        total += shifts[i]
        res.append(chr(ord(s[i]) + total))             # SAI: không modulo
    return "".join(reversed(res))


def bug_off_by_one(s, shifts):                         # chữ i chịu shifts[j] với j > i (lệch một)
    n = len(s)
    res, total = [""] * n, 0
    for i in range(n - 1, -1, -1):
        res[i] = shift_char(s[i], total)               # SAI: cộng shifts[i] sau khi dùng
        total = (total + shifts[i]) % 26
    return "".join(res)


def bug_int32(s, shifts):                              # mô phỏng int 32-bit tràn khi cộng dồn không mod
    def wrap(v):
        return ((v + 2 ** 31) % 2 ** 32) - 2 ** 31
    res, total = [""] * len(s), 0
    for i in range(len(s) - 1, -1, -1):
        total = wrap(total + shifts[i])
        res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
    return "".join(res)


BUGS = [bug_only_self, bug_prefix, bug_no_mod, bug_off_by_one, bug_int32]
rng = random.Random(1)
tests = [random_case(rng, rng.randint(2, 6), 30) for _ in range(500)]
tests.append(("abcde", [10 ** 9] * 5))
for bug in BUGS:
    bad = next(((s, sh) for s, sh in tests if bug(s, sh) != shifting_suffix(s, sh)), None)
    if bad is None:                                    # int32 chỉ lộ khi tổng tràn: dùng ca lớn
        bad = ("a" * 5, [10 ** 9] * 5)
    got = bug(*bad)
    assert got != shifting_suffix(*bad), bug.__name__
    print(f"{bug.__name__:16s} sai trên {bad}: ra {got!r}, đúng {shifting_suffix(*bad)!r}")

bug_only_self    sai trên ('szy', [2, 8, 3]): ra 'uhb', đúng 'fkb'
bug_prefix       sai trên ('szy', [2, 8, 3]): ra 'ujl', đúng 'fkb'
bug_no_mod       sai trên ('szy', [2, 8, 3]): ra '\x80\x85|', đúng 'fkb'
bug_off_by_one   sai trên ('szy', [2, 8, 3]): ra 'dcy', đúng 'fkb'
bug_int32        sai trên ('abcde', [1000000000, 1000000000, 1000000000, 1000000000, 1000000000]): ra 'mbqbq', đúng 'ixmbq'


### D. Vét cạn mọi đầu vào nhỏ

Mọi chuỗi độ dài `1..3` trên bảng chữ `{a, y, z}` và mọi `shifts` có phần tử trong `{0, 1, 2, 25, 26, 27}` (`3 · 6 + 9 · 36 + 27 · 216 = 6 174` ca). Oracle **mô phỏng từng lần dịch** (với `x ≤ 27` thì nhanh). Mọi cài đặt phải khớp.

In [22]:
t0 = time.perf_counter()
cnt = 0
for L in range(1, 4):
    for s_t in product("ayz", repeat=L):
        s = "".join(s_t)
        for sh in product((0, 1, 2, 25, 26, 27), repeat=L):
            want = oracle(s, list(sh))
            for name in IMPLS:
                assert IMPLS[name](s, list(sh)) == want, (name, s, sh)
            cnt += 1
print(cnt, "ca vét cạn: 8 cài đặt khớp oracle mô phỏng từng bước;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 3 * 6 + 9 * 36 + 27 * 216

6174 ca vét cạn: 8 cài đặt khớp oracle mô phỏng từng bước; 0.2 s


### E. Ngẫu nhiên có seed

Chuỗi độ dài `1..40`, `shifts` tới `60` (oracle mô phỏng từng bước còn chạy được), và ca lớn hơn (độ dài `1..300`, `shifts` tới `10^9`) so với cách `brute` dùng `% 26`.

In [23]:
rng = random.Random(20240848)
cnt = 0
for _ in range(2000):
    n = rng.randint(1, 40)
    s, sh = random_case(rng, n, rng.choice([1, 5, 25, 26, 60]))
    check(s, sh)
    cnt += 1
for _ in range(500):
    n = rng.randint(1, 300)
    s, sh = random_case(rng, n, rng.choice([100, 10 ** 5, 10 ** 9]))
    want = shifting_brute(s, sh)
    check(s, sh, want)
    cnt += 1
print(cnt, "ca ngẫu nhiên: mọi cài đặt trùng oracle / brute")

# tính chất: (1) thêm 26 vào mọi shifts[i] không đổi; (2) dịch tất cả chữ cái +t: kết quả dịch +t; (3) tổng shifts bất biến theo hoán vị khi mọi chữ cái giống nhau không đúng -> chỉ kiểm 1-2
for _ in range(300):
    n = rng.randint(1, 30)
    s, sh = random_case(rng, n, 10 ** 9)
    base = Solution().shiftingLetters(s, sh)
    assert Solution().shiftingLetters(s, [x + 26 * rng.randint(0, 5) for x in sh]) == base
    t = rng.randint(0, 25)
    s2 = "".join(shift_char(c, t) for c in s)
    assert Solution().shiftingLetters(s2, sh) == "".join(shift_char(c, t) for c in base)
print("tính chất: +26 không đổi; dịch chuỗi gốc +t thì kết quả dịch +t — OK")

2500 ca ngẫu nhiên: mọi cài đặt trùng oracle / brute
tính chất: +26 không đổi; dịch chuỗi gốc +t thì kết quả dịch +t — OK


### F. Giới hạn lớn nhất

`n = 10^5`, `shifts[i] = 10^9`: tổng hậu tố tới `10^14`. Kiểm tra mọi cách nhanh khớp nhau và khớp công thức đóng.

In [24]:
n = 10 ** 5
fast = ["suffix", "diff", "accumulate", "Solution", "SolutionSuffix", "SolutionDiff", "SolutionAccumulate"]
rng = random.Random(777)
cases = {
    "ngẫu nhiên": random_case(rng, n),
    "toàn z, shifts = 10^9": ("z" * n, [10 ** 9] * n),
    "toàn a, shifts = 0": ("a" * n, [0] * n),
    "toàn a, shifts = 26": ("a" * n, [26] * n),
    "chỉ shifts cuối khác 0": ("m" * n, [0] * (n - 1) + [7]),
    "chỉ shifts đầu khác 0": ("m" * n, [7] + [0] * (n - 1)),
}
for name, (s, sh) in cases.items():
    ref = shifting_suffix(s, sh)
    for impl in fast:
        assert IMPLS[impl](s, sh) == ref, (name, impl)
    print(f"{name:26s}: {ref[:8]}...{ref[-4:]} (7 cài đặt khớp)")
# công thức đóng cho trường hợp toàn 10^9: chữ j (0-indexed) chịu (n - j) lần 10^9
s, sh = cases["toàn z, shifts = 10^9"]
expected = "".join(chr((25 + (n - j) * 10 ** 9) % 26 + 97) for j in range(n))
assert Solution().shiftingLetters(s, sh) == expected
assert Solution().shiftingLetters(*cases["chỉ shifts cuối khác 0"]) == "t" * n
assert Solution().shiftingLetters(*cases["chỉ shifts đầu khác 0"]) == "t" + "m" * (n - 1)
print("công thức đóng khớp")

ngẫu nhiên                : otwlxkvv...wfdh (7 cài đặt khớp)
toàn z, shifts = 10^9     : vjxlznbp...vjxl (7 cài đặt khớp)


toàn a, shifts = 0        : aaaaaaaa...aaaa (7 cài đặt khớp)
toàn a, shifts = 26       : aaaaaaaa...aaaa (7 cài đặt khớp)


chỉ shifts cuối khác 0    : tttttttt...tttt (7 cài đặt khớp)
chỉ shifts đầu khác 0     : tmmmmmmm...mmmm (7 cài đặt khớp)


công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ dịch chữ `i` đi `shifts[i]` (quên cộng dồn)

Đọc nhanh "dịch `shifts[i]`" thành "dịch chữ thứ `i`". Đề nói dịch **`i + 1` chữ đầu**: các phép dịch **chồng** lên chữ ở đầu.

In [25]:
s, sh = ex1
print("chỉ shifts[i]:", bug_only_self(s, sh), "| đúng:", Solution().shiftingLetters(s, sh))
assert bug_only_self(s, sh) == "dgl" and Solution().shiftingLetters(s, sh) == "rpl"

chỉ shifts[i]: dgl | đúng: rpl


**Vì sao sai:** chữ `0` chịu cả ba phép (`3 + 5 + 9`). **Đúng:** dịch bằng **tổng hậu tố** `Σ_{i ≥ j} shifts[i]`.

### Lỗi 2: cộng tiền tố thay vì hậu tố

Chữ `j` chịu các phép `i ≥ j` (phép **sau** nó), không phải `i ≤ j`. Cộng tiền tố cho kết quả hoàn toàn khác.

In [26]:
print("tiền tố:", bug_prefix(*ex1), "| đúng:", Solution().shiftingLetters(*ex1))
assert bug_prefix(*ex1) != "rpl"

tiền tố: djt | đúng: rpl


**Vì sao sai:** phép thứ `i` phủ vị trí `0..i`, tức các chữ **đứng trước** nó; do đó chữ đầu tiên nhận từ **mọi** phép. **Đúng:** duyệt ngược `i` từ `n − 1` về `0` (hậu tố).

### Lỗi 3: quên `% 26`

Dịch `'z'` đi `1` bước phải cho `'a'`, không phải `'{'` (ký tự sau `'z'` trong bảng ASCII).

In [27]:
print("không mod:", repr(bug_no_mod("z", [1])), "| đúng:", repr(Solution().shiftingLetters("z", [1])))
print("ví dụ 1:", repr(bug_no_mod(*ex1)), "| đúng:", repr(Solution().shiftingLetters(*ex1)))
assert bug_no_mod("z", [1]) == "{" and Solution().shiftingLetters("z", [1]) == "a"
assert "".join(c for c in bug_no_mod("abc", [10 ** 4] * 3) if not ("a" <= c <= "z"))

không mod: '{' | đúng: 'a'
ví dụ 1: 'rpl' | đúng: 'rpl'


**Vì sao sai:** bảng chữ cái là **vòng tròn** 26 phần tử. **Đúng:** `(ord(c) - ord('a') + total) % 26 + ord('a')`.

### Lỗi 4: lệch một (`i` thay vì `i + 1`)

Nếu cộng `shifts[i]` **sau khi** dùng `total` cho chữ `i` thì chữ cuối nhận `0` (thiếu `shifts[n−1]`).

In [28]:
print("lệch một:", bug_off_by_one(*ex1), "| đúng:", Solution().shiftingLetters(*ex1))
assert bug_off_by_one("a", [5]) == "a" and Solution().shiftingLetters("a", [5]) == "f"

lệch một: okc | đúng: rpl


**Vì sao sai:** phép `shifts[i]` phủ cả vị trí `i` (vì phủ `i + 1` chữ đầu). **Đúng:** cập nhật `total` **trước** khi dịch chữ `i`.

### Lỗi 5: tràn số nguyên ở ngôn ngữ có `int` 32-bit

Tổng hậu tố tới `10^14` vượt `2^31 − 1`. Python không tràn nhưng có thể **mô phỏng**: nếu cộng dồn không `% 26` trong `int` 32-bit, kết quả sai. Cách an toàn: `% 26` **sau mỗi phép cộng** (giữ `total < 26 + 10^9 < 2^31`) hoặc dùng `long`.

In [29]:
s, sh = "a" * 5, [10 ** 9] * 5
print("mô phỏng int32 không mod:", bug_int32(s, sh), "| đúng:", Solution().shiftingLetters(s, sh))
print("tổng thật:", sum(sh), "| 2^31 - 1 =", 2 ** 31 - 1)
assert bug_int32(s, sh) != Solution().shiftingLetters(s, sh)

mô phỏng int32 không mod: maoym | đúng: iwkym
tổng thật: 5000000000 | 2^31 - 1 = 2147483647


**Vì sao sai:** `5·10^9 > 2^31`, bị cắt. **Đúng:** `total = (total + shifts[i]) % 26` mỗi bước.

### Lỗi 6 (bonus): dịch từng lần theo `shifts[i]` bằng vòng lặp `x` lần

`for _ in range(x)` với `x` tới `10^9` không bao giờ xong. Dịch `x` bước bằng **một** phép `(k + x) % 26`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "nhiều thao tác, mỗi thao tác áp lên **một tiền tố / đoạn**; hỏi giá trị cuối của từng phần tử" → **tổng hậu tố/tiền tố** hoặc **mảng hiệu** (difference array). Nếu thao tác **cộng được** và có **modulo** thì có thể gộp.

**Mẹo:**

1. Hỏi: phần tử `j` chịu những thao tác nào? Ở đây `i ≥ j`: hậu tố.
2. Phép cộng modulo gộp được: tính **tổng rồi `% 26`**.
3. Nhớ `% 26` ngay khi cộng dồn để tránh tràn ở ngôn ngữ 32-bit.
4. Đoạn tổng quát `[l, r]` → mảng hiệu: `diff[l] += x`, `diff[r + 1] -= x`.
5. `chr`/`ord` cần chuyển sang chỉ số `0..25`.
6. Oracle: mô phỏng **từng lần dịch** cho `x` nhỏ.

**Họ bài cùng pattern:** Shifting Letters II (2381, đoạn bất kỳ), Range Addition (370), Corporate Flight Bookings (1109), Car Pooling (1094), Product of Array Except Self (238).

## 15. Biến thể

1. **Shifting Letters II (2381):** mỗi thao tác `[start, end, direction]` dịch tiến (`1`) hoặc lùi (`0`) các chữ trong `[start, end]` — dùng mảng hiệu.
2. **Dịch lùi:** `shifts[i]` âm hoặc dịch ngược.
3. **Dịch `suffix` thay vì `prefix`** (đảo vai trò tiền tố/hậu tố).
4. **Giải mã:** cho chuỗi cuối và `shifts`, tìm chuỗi gốc.
5. **Bảng chữ cái khác** (hoa/thường, hoặc chữ số vòng 10).

Cài đặt biến thể 1 và 4 và đối chiếu với brute:

In [30]:
def shifting_letters_ii(s, shifts):
    """LeetCode 2381: shifts[i] = [start, end, direction]; direction 1 = dịch tiến, 0 = dịch lùi."""
    n = len(s)
    diff = [0] * (n + 1)
    for start, end, d in shifts:
        v = 1 if d == 1 else -1
        diff[start] += v
        diff[end + 1] -= v
    out, cur = [], 0
    for j in range(n):
        cur += diff[j]
        out.append(chr((ord(s[j]) - 97 + cur) % 26 + 97))
    return "".join(out)


def shifting_letters_ii_brute(s, shifts):
    a = [ord(c) - 97 for c in s]
    for start, end, d in shifts:
        for j in range(start, end + 1):
            a[j] = (a[j] + (1 if d == 1 else -1)) % 26
    return "".join(chr(v + 97) for v in a)


assert shifting_letters_ii("abc", [[0, 1, 0], [1, 2, 1], [0, 2, 1]]) == "ace"
assert shifting_letters_ii("dztz", [[0, 0, 0], [1, 1, 1]]) == "catz"
rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 15)
    s = "".join(rng.choice("abcxyz") for _ in range(n))
    qs = []
    for _ in range(rng.randint(0, 10)):
        l = rng.randrange(n); r = rng.randint(l, n - 1)
        qs.append([l, r, rng.randint(0, 1)])
    assert shifting_letters_ii(s, qs) == shifting_letters_ii_brute(s, qs)
print("Shifting Letters II (mảng hiệu) khớp brute trên 500 ca")


def unshift(result, shifts):
    """Biến thể 4: khôi phục chuỗi gốc từ chuỗi cuối và shifts."""
    n = len(result)
    out, total = [""] * n, 0
    for i in range(n - 1, -1, -1):
        total = (total + shifts[i]) % 26
        out[i] = chr((ord(result[i]) - 97 - total) % 26 + 97)
    return "".join(out)


for _ in range(300):
    s, sh = random_case(rng, rng.randint(1, 20), 10 ** 9)
    assert unshift(Solution().shiftingLetters(s, sh), sh) == s
print("giải mã: unshift(shiftingLetters(s, sh), sh) == s trên 300 ca")

Shifting Letters II (mảng hiệu) khớp brute trên 500 ca
giải mã: unshift(shiftingLetters(s, sh), sh) == s trên 300 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dịch từng tiền tố | Với mỗi `i`, dịch `i + 1` chữ đầu | `O(n²)` | `O(n)` | Oracle; `n` nhỏ |
| 2. Tổng hậu tố duyệt ngược | `total` cộng dồn từ phải sang trái | `O(n)` | `O(n)` | **Đáp án chính** |
| 3. Mảng hiệu | `diff[0] += x; diff[i+1] -= x`, tổng tiền tố | `O(n)` | `O(n)` | Đoạn bất kỳ (bài II) |
| 4. `accumulate` | `S[j] = tổng − tiền tố trước j` | `O(n)` | `O(n)` | Ngắn gọn |

Cả bốn khớp oracle mô phỏng từng bước trên 6 174 ca vét cạn và hàng nghìn ca ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
print(shift_char("y", 3), shift_char("a", 52), shift_char("m", 10 ** 9))
print(Solution().shiftingLetters("ab", [1, 1]), Solution().shiftingLetters("zz", [1, 25]), Solution().shiftingLetters("abc", [26, 26, 26]))

b a y
cc zy abc


### Bài 2 (tính tay)

`s = "xyz"`, `shifts = [2, 3, 4]`.

- (a) Tính tổng hậu tố `S[j]` cho từng chữ.
- (b) Chữ nào quay vòng qua `z`?
- (c) Kết quả cuối là gì?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tổng hậu tố:

In [32]:
def suffix_todo(s, shifts):
    n = len(s)
    res = [""] * n
    total = 0
    for i in range(n - 1, -1, -1):
        pass            # TODO 1: total = (total + shifts[i]) % 26
        pass            # TODO 2: res[i] = chữ s[i] dịch total bước (nhớ quay vòng)
    return "".join(res)


# Khi làm xong, bỏ comment để kiểm tra:
# assert suffix_todo("abc", [3, 5, 9]) == "rpl" and suffix_todo("aaa", [1, 2, 3]) == "gfd"

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mảng hiệu**:

In [33]:
class MySolution:
    def shiftingLetters(self, s, shifts):
        pass            # TODO


# Khi làm xong:
# assert MySolution().shiftingLetters("abc", [3, 5, 9]) == "rpl" and MySolution().shiftingLetters("aaa", [1, 2, 3]) == "gfd"

### Bài 5 (mini challenge): dịch hậu tố

Phiên bản đảo: `shifts[i] = x` dịch các chữ từ vị trí `i` **đến cuối chuỗi** (hậu tố `s[i:]`) thêm `x` lần. Viết `shifting_suffixes(s, shifts)` trong `O(n)` và kiểm tra bằng mô phỏng từng lần dịch.

In [34]:
def shifting_suffixes(s, shifts):
    pass            # TODO


# Khi làm xong:
# assert shifting_suffixes("abc", [3, 5, 9]) == "djt"

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `shift_char(c, x)` là `(k + x) % 26`.
- Mức 2: `'y'` (24) + 3 = 27 % 26 = 1 → `'b'`; `'a'` + 52 = `'a'`; `10^9 % 26 = 12` → `'m' + 12 = 'y'`.
- Mức 3: `b a y`; `"ab"`, `[1,1]` → `S = [2,1]` → `"cc"`; `"zz"`, `[1,25]` → chữ 0: 26 → `'z'`; chữ 1: 25 → `'y'`; `"abc"`, `[26,26,26]` → `"abc"`.

**Bài 2**

- Mức 1: `S[2] = 4`, `S[1] = 3 + 4 = 7`, `S[0] = 2 + 7 = 9`.
- Mức 2: `x` (23) + 9 = 32 % 26 = 6 → `g`; `y` (24) + 7 = 31 % 26 = 5 → `f`; `z` (25) + 4 = 29 % 26 = 3 → `d`.
- Mức 3: cả ba chữ đều quay vòng; kết quả `"gfd"`.

**Bài 3**

- Mức 1: `total = (total + shifts[i]) % 26`.
- Mức 2: `res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)`.
- Mức 3: cập nhật `total` **trước** khi dịch chữ `i`.

**Bài 4**

- Mức 1: `diff[0] += x; diff[i+1] -= x`.
- Mức 2: tổng tiền tố của `diff` là độ dịch.
- Mức 3: xem Cách 3.

**Bài 5**

- Mức 1: chữ `j` chịu `shifts[i]` với `i ≤ j`: tiền tố.
- Mức 2: duyệt xuôi, cộng dồn `total`.
- Mức 3: kiểm bằng mô phỏng.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
assert (shift_char("y", 3), shift_char("a", 52), shift_char("m", 10 ** 9)) == ("b", "a", "y")
assert Solution().shiftingLetters("ab", [1, 1]) == "cc" and Solution().shiftingLetters("abc", [26, 26, 26]) == "abc"
assert Solution().shiftingLetters("zz", [1, 25]) == oracle("zz", [1, 25]) == "zy"
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `S = [9, 7, 4]`. (b) Cả `x` (23 + 9 = 32), `y` (24 + 7 = 31), `z` (25 + 4 = 29) đều vượt 25 và quay vòng. (c) `"gfd"`.

In [36]:
s, sh = "xyz", [2, 3, 4]
S = [sum(sh[j:]) for j in range(3)]
assert S == [9, 7, 4]
assert [(ord(c) - 97 + x) for c, x in zip(s, S)] == [32, 31, 29]
assert Solution().shiftingLetters(s, sh) == "gfd" == oracle(s, sh)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [37]:
def suffix_answer(s, shifts):
    n = len(s)
    res = [""] * n
    total = 0
    for i in range(n - 1, -1, -1):
        total = (total + shifts[i]) % 26
        res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
    return "".join(res)


assert suffix_answer("abc", [3, 5, 9]) == "rpl" and suffix_answer("aaa", [1, 2, 3]) == "gfd"
rng = random.Random(3)
for _ in range(500):
    s, sh = random_case(rng, rng.randint(1, 15), 60)
    assert suffix_answer(s, sh) == oracle(s, sh)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def shiftingLetters(self, s, shifts):
        n = len(s)
        diff = [0] * (n + 1)
        for i, x in enumerate(shifts):
            diff[0] += x
            diff[i + 1] -= x
        out, cur = [], 0
        for j in range(n):
            cur += diff[j]
            out.append(chr((ord(s[j]) - 97 + cur) % 26 + 97))
        return "".join(out)


assert MySolutionAnswer().shiftingLetters("abc", [3, 5, 9]) == "rpl" and MySolutionAnswer().shiftingLetters("aaa", [1, 2, 3]) == "gfd"
rng = random.Random(4)
for _ in range(500):
    s, sh = random_case(rng, rng.randint(1, 40), 10 ** 9)
    assert MySolutionAnswer().shiftingLetters(s, sh) == Solution().shiftingLetters(s, sh)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

Chữ `j` chịu `shifts[i]` với `i ≤ j` (hậu tố bắt đầu tại `i` phủ `j ≥ i`): tổng **tiền tố** `P[j]`.

In [39]:
def shifting_suffixes_answer(s, shifts):
    out, total = [], 0
    for c, x in zip(s, shifts):
        total = (total + x) % 26
        out.append(chr((ord(c) - 97 + total) % 26 + 97))
    return "".join(out)


def shifting_suffixes_oracle(s, shifts):
    nxt = {chr(c): chr(c + 1) for c in range(97, 122)}
    nxt["z"] = "a"
    a = list(s)
    for i, x in enumerate(shifts):
        for _ in range(x):
            for j in range(i, len(a)):
                a[j] = nxt[a[j]]
    return "".join(a)


assert shifting_suffixes_answer("abc", [3, 5, 9]) == "djt"
rng = random.Random(5)
for _ in range(500):
    s, sh = random_case(rng, rng.randint(1, 12), 40)
    assert shifting_suffixes_answer(s, sh) == shifting_suffixes_oracle(s, sh)
print("Bài 5 OK: khớp mô phỏng từng lần dịch trên 500 ca")

Bài 5 OK: khớp mô phỏng từng lần dịch trên 500 ca


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (thao tác áp lên tiền tố → tổng hậu tố, giữ mod):**

```python
class Solution:
    def shiftingLetters(self, s: str, shifts: List[int]) -> str:
        res = [""] * len(s)
        total = 0
        for i in range(len(s) - 1, -1, -1):
            total = (total + shifts[i]) % 26               # chữ i chịu shifts[i], shifts[i+1], ...
            res[i] = chr((ord(s[i]) - 97 + total) % 26 + 97)
        return "".join(res)
```

**Checklist trước khi submit:**

- [ ] Chữ `j` chịu `shifts[i]` với **`i ≥ j`** (hậu tố), không phải tiền tố, không chỉ `shifts[j]`?
- [ ] Cập nhật `total` **trước** khi dịch chữ `i` (phép `shifts[i]` phủ cả chữ `i`)?
- [ ] Có `% 26` để quay vòng `z → a` và để tránh tràn số?
- [ ] Không lặp `x` lần (với `x` tới `10^9`); dùng một phép `(k + x) % 26`?
- [ ] Đã thử `"abc",[3,5,9]` → `"rpl"`, `"aaa",[1,2,3]` → `"gfd"`, `shifts` bội `26`, `"z"`?
- [ ] Đúng tên `class Solution`, `shiftingLetters(self, s, shifts)`, không in debug.